# LAB 2 — ONE-NOTEBOOK PRESENTATION FLOW
### Nhóm 4 · CIFAR-10 · Pre-trained VGG16 & DenseNet121

**Mục đích:** một notebook duy nhất để **thuyết trình từ đầu đến cuối**, sử dụng bằng chứng thực nghiệm đã có trong repository. Không thay thế 6 notebook chuyên môn `01`–`06` hay các module `src/`.

**Thông điệp trọng tâm:** Nạp mô hình pre-trained ImageNet → thay classifier cho 10 lớp CIFAR-10 → classifier-only / fine-tuning → chạy 5 thí nghiệm → monitoring → chọn checkpoint theo validation → đánh giá test.

**Thời lượng gợi ý: 15–18 phút.** Người trình bày chỉ cần mở notebook này, chạy **Run All** trong môi trường đã cài `pandas`, `matplotlib`, `Pillow`, `IPython`. Chế độ mặc định là **REVIEW ONLY**: **không train, không download, không tự tạo metric**.

> **Nguồn số liệu:** `outputs/experiment_summary.csv`, `outputs/runs/<run>/history.csv`, `outputs/runs/E5_cpu_1epoch/metrics.json`, báo cáo `reports/Lab2_Report.md`, code `src/` và hình `reports/figures/`. Phần live demonstration chỉ thực hiện khi chủ động bật cờ ở cuối notebook.

**Chỉ dẫn trình bày:** Các đoạn “🎤 Script” là lời dẫn trực tiếp; có thể đọc hoặc diễn đạt lại. Các đoạn “⚠ Giới hạn” là nội dung phải nói đúng khi giảng viên phản biện.


## 0. Đối chiếu repository & kiểm toán trước khi trình bày

Notebook tổng hợp được lập sau khi rà soát **nhánh `feature/data`** tại commit `40b0191f95bec13b89f1a0bbdde86c72078174dc` (20/11/2026).

| Nguồn | Vai trò trong bài nói |
|---|---|
| `01_environment_pretrained.ipynb` | Môi trường, ImageNet weights, khảo sát kiến trúc |
| `02_model_adaptation.ipynb` | Classifier 10 lớp, freeze và fine-tune |
| `03_data_exploration_preprocessing.ipynb` | EDA, split, transforms, DataLoader |
| `04_training_experiments.ipynb` | Training loop, ma trận E1–E5, checkpoint |
| `05_model_monitoring.ipynb` | TensorBoard, history, gap, overfitting caveat |
| `06_evaluation_results.ipynb` | So sánh, chọn model, test, kết luận |

**Audit quan trọng:**
- Sáu notebook bao phủ đúng trục chính của đề Lab 2. Để kể chuyện tự nhiên, phần **Data** được trình bày **trước Model**, dù notebook chuyên môn mang số `03`.
- Trong các notebook nguồn: `MODE="review"` đọc kết quả lịch sử; `MODE="full"` mới chạy dữ liệu/huấn luyện thật. **Không gọi review là thực nghiệm vừa chạy.**
- Branch `feature/data` không còn `03_eda_pca.ipynb` và `02_results_analysis.ipynb` như nhánh cũ. **PCA không nằm trong core Lab 2**; không ghép kết quả PCA vào flow chính.
- Mỗi E1–E5 đã hoàn thành **1 epoch, seed 42, CPU**. E5 phiên bản **10 epoch còn thiếu metrics**.
- Screenshot TensorBoard là **minh chứng đã lưu**, không thay thế event logs mới. Không dùng một điểm/epoch để kết luận convergence hay overfitting.
- Một số notebook nguồn mang kernelspec khác nhau; sử dụng **cùng môi trường Python của dự án** khi demo. Nên kiểm tra trước các tệp hình và output cục bộ.
- Nhánh này thay đổi/xóa nhiều notebook so với `main`; khi merge cần review diff, **không đồng nhất trạng thái branch với main**.


In [ ]:
from pathlib import Path
import json, ast
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Code, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src").is_dir() and (p / "configs").is_dir()
             and (p / "outputs").is_dir()), None)
assert ROOT is not None, "Mở notebook từ thư mục project Lab2_DL_Group4."
FIG = ROOT / "reports" / "figures"
OUT = ROOT / "outputs"
SUMMARY = OUT / "experiment_summary.csv"
RUNS = ["E1_vgg16", "E2_densenet121", "E3_learning_rate",
        "E4_batch_size", "E5_cpu_1epoch"]
assert SUMMARY.exists(), f"Thiếu số liệu thực nghiệm: {SUMMARY}"
raw_summary = pd.read_csv(SUMMARY)
results = raw_summary.loc[
    raw_summary["experiment_name"].isin(RUNS)
    & raw_summary["best_val_accuracy"].notna()
].copy()
assert len(results) == 5, "Cần đủ 5 run E1–E5 đã hoàn thành."
results["val_accuracy_pct"] = results["best_val_accuracy"] * 100
results["duration_minutes"] = results["training_time_sec"] / 60
results = results.set_index("experiment_name").loc[RUNS].reset_index()

def show_figure(filename, width=900):
    p = FIG / filename
    if p.is_file():
        display(Image(filename=str(p), width=width))
        print("Nguồn hình đã lưu:", p.relative_to(ROOT))
    else:
        print("CHƯA CÓ HÌNH TẠI MÁY:", p.relative_to(ROOT))

def show_function(path, function, limit=65):
    p = ROOT / path
    if not p.is_file():
        print("Không tìm thấy:", p)
        return
    source = p.read_text(encoding="utf-8-sig")
    tree = ast.parse(source)
    node = next((n for n in tree.body
                 if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
                 and n.name == function), None)
    if node is None:
        print(f"Không tìm thấy {function} trong {path}")
        return
    lines = source.splitlines()
    excerpt = lines[node.lineno-1:min(node.end_lineno, node.lineno-1+limit)]
    display(Code("\n".join(excerpt), language="python"))
    if node.end_lineno - node.lineno + 1 > limit:
        print(f"(Hiển thị {limit} dòng đầu; xem đầy đủ tại {path})")

print("Dự án:", ROOT)
print("Chế độ: REVIEW — chỉ đọc artifact đã lưu, không training.")
display(results[["experiment_name", "model_name", "strategy", "epochs", "val_accuracy_pct"]])


### Audit tự động: có đủ minh chứng để demo không?


In [ ]:
required = [
    *[f"notebooks/{n}" for n in [
        "01_environment_pretrained.ipynb", "02_model_adaptation.ipynb",
        "03_data_exploration_preprocessing.ipynb",
        "04_training_experiments.ipynb", "05_model_monitoring.ipynb",
        "06_evaluation_results.ipynb"]],
    "src/data.py", "src/models.py", "src/train.py", "src/evaluate.py",
    "outputs/experiment_summary.csv",
    "outputs/runs/E5_cpu_1epoch/metrics.json",
    "reports/figures/eda_class_distribution.png",
    "reports/figures/cifar10_samples.png",
    "reports/figures/tensorboard_comparison.png",
]
audit = pd.DataFrame([{"Tệp": p, "Có tại máy": (ROOT/p).exists()} for p in required])
display(audit)
print("Tệp thiếu:", int((~audit["Có tại máy"]).sum()))
print("Hàng chưa hoàn thành trong summary:")
display(raw_summary[raw_summary["best_val_accuracy"].isna()][["experiment_name", "epochs"]])


## 1. Đặt vấn đề & mục tiêu — 1 phút

**Câu hỏi nghiên cứu:** Có thể tận dụng kiến thức hình ảnh của mạng CNN đã học trên **ImageNet (1.000 lớp)** để phân loại **CIFAR-10 (10 lớp)** không? Khi thay kiến trúc, learning rate, batch size hoặc mở fine-tuning thì kết quả thay đổi thế nào?

**Yêu cầu đề:** thử nhiều pre-trained architectures; khảo sát layers/parameters; thay classifier; freeze/fine-tuning; transforms/DataLoader; loss/optimizer/training loop; hyperparameter experiments; TensorBoard; đánh giá test.

🎤 **Script:** “Kính chào thầy. Lab 2 của nhóm tập trung vào **transfer learning**, chứ không phải xây mạng từ đầu hay tối ưu accuracy bằng mọi giá. Chúng em chọn VGG16 và DenseNet121 pre-trained trên ImageNet, chuyển chúng sang CIFAR-10, rồi thiết kế các phép thử có kiểm soát để đánh giá. Em sẽ trình bày dữ liệu, mô hình, cách huấn luyện, monitoring, kết quả và giới hạn.”

**Chuyển ý:** “Trước tiên phải đảm bảo dữ liệu đưa vào mô hình được chuẩn bị đúng.”


## 2. Dataset & EDA — 2 phút

- **CIFAR-10:** 60.000 ảnh RGB, **32×32**, 10 lớp: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.
- **50.000 train gốc / 10.000 test**; train gốc được tách thành **45.000 train + 5.000 validation**, seed 42.
- Mỗi lớp có **5.000 ảnh train gốc**, **1.000 ảnh test**. Validation sau `random_split` **không stratify** nên số lượng từng lớp dao động (471–532 theo notebook nguồn).
- EDA giúp thấy ảnh nhỏ, đa dạng góc chụp, nền và màu sắc. Có kiểm tra trùng byte, nhưng **không có nghĩa loại trừ mọi kiểu data leakage**.

🎤 **Script:** “Dữ liệu CIFAR-10 có 10 nhãn và được cân bằng ở train gốc/test. Chúng em tách 10% train gốc làm validation bằng seed cố định, đồng thời giữ tập test riêng để đánh giá cuối. Phần EDA kiểm tra phân bố nhãn, ảnh mẫu và đặc trưng màu sắc trước khi áp dụng transfer learning.”

**⚠ Giới hạn:** Không được nói đã dùng phân tầng (stratified) hay đảm bảo ‘zero leakage tuyệt đối’ chỉ từ kiểm tra hash.


In [ ]:
show_figure("eda_class_distribution.png", width=800)
show_figure("cifar10_samples.png", width=780)


### 2.1. Ảnh hưởng của resize và normalization

Pipeline trong `src/data.py` dùng:
- **Resize** 32×32 → **224×224** (số pixel tăng **49 lần**; **không tạo thêm chi tiết thực**).
- **Train:** RandomHorizontalFlip(p=0.5), RandomCrop với padding=4, ToTensor, Normalize.
- **Validation/test:** chỉ Resize, ToTensor, Normalize — không augmentation ngẫu nhiên.
- Normalize theo mean/std **ImageNet**: `(0.485, 0.456, 0.406)` / `(0.229, 0.224, 0.225)`.
- Sau xử lý, batch ảnh có shape **[B, 3, 224, 224]**.

🎤 **Script:** “Do VGG16 và DenseNet121 sử dụng trọng số pre-trained của ImageNet, nhóm đưa ảnh về kích thước và chuẩn hóa phù hợp. Augmentation chỉ dùng ở train để bổ sung biến thiên, trong khi validation và test phải ổn định để so sánh. Resize tăng số pixel nhưng cũng làm tăng thời gian xử lý.”

**Chứng minh bằng code gốc:**


In [ ]:
show_function("src/data.py", "get_transforms", limit=40)
show_figure("eda_resize_effect.png", width=760)


## 3. Pre-trained CNN: VGG16 và DenseNet121 — 2 phút

**Pre-trained** nghĩa là các mô hình đã được học trên ImageNet trước khi nhóm dùng cho CIFAR-10.

| Kiến trúc | Điểm chính | Bộ phân loại gốc |
|---|---|---|
| **VGG16** | Các khối convolution/pooling tuần tự, classifier nhiều fully-connected layers | 1.000 lớp |
| **DenseNet121** | Dense connections giúp tái sử dụng feature maps từ các layer trước | 1.000 lớp |

Nhóm dùng `torchvision.models` với `*_Weights.DEFAULT`, **thay classifier cuối ra 10 logits**.

🎤 **Script:** “Hai kiến trúc đều có backbone trích xuất đặc trưng và classifier dự đoán nhãn. VGG16 có mạng phân loại lớn, DenseNet121 sử dụng dense connections. Thay vì huấn luyện toàn bộ từ đầu, chúng em tận dụng các đặc trưng đã học trên ImageNet và thay đầu ra để phù hợp 10 lớp.”

**Câu hỏi thầy dễ hỏi:** Vì sao output là **logits**, không thêm Softmax? Vì `CrossEntropyLoss` nhận trực tiếp logits và xử lý biến đổi cần thiết bên trong.


### 3.1. Freeze backbone, classifier-only và fine-tuning

| Cấu hình đã chạy | Tổng parameter | Trainable parameter |
|---|---:|---:|
| VGG16 — classifier_only | 134.301.514 | 119.586.826 |
| DenseNet121 — classifier_only | 6.964.106 | 10.250 |
| DenseNet121 — finetune_last_layers | 6.964.106 | 2.170.378 |

**Classifier-only:** backbone `requires_grad=False`, classifier `requires_grad=True`. Với VGG16, **toàn bộ classifier** (gồm nhiều lớp FC) được huấn luyện.

**Fine-tuning E5:** ngoài classifier, nhóm mở `features.denseblock4` và `features.norm5` của DenseNet121.

🎤 **Script:** “Classifier-only giữ nguyên bộ trích xuất đặc trưng và học bộ phân loại mới. Fine-tuning mở thêm lớp cuối để điều chỉnh biểu diễn cho CIFAR-10. Nhóm so sánh hai chiến lược trên DenseNet121 để xem khả năng thích nghi của backbone có giúp tăng độ chính xác sau một epoch hay không.”

⚠ **Công bằng thực nghiệm:** VGG16 và DenseNet121 khác nhau rất lớn về trainable parameters, nên E1–E2 là **so sánh cấu hình thực tế**, không phải phép thử tách riêng mọi nguyên nhân.


In [ ]:
params = pd.DataFrame([
    ["VGG16 classifier-only", 134301514, 119586826],
    ["DenseNet121 classifier-only", 6964106, 10250],
    ["DenseNet121 fine-tune", 6964106, 2170378],
], columns=["Cấu hình", "Tổng parameters", "Trainable parameters"])
params["Tỷ lệ trainable (%)"] = (100*params["Trainable parameters"]/params["Tổng parameters"]).round(2)
display(params)
show_function("src/models.py", "_replace_classifier", limit=28)
show_function("src/models.py", "_configure_trainable_parameters", limit=52)


## 4. Training pipeline — 2 phút

Một vòng huấn luyện:
1. `model.train()`; giữ BatchNorm đã freeze ở chế độ eval để không cập nhật running statistics.
2. Mỗi batch: chuyển ảnh/nhãn sang device → `optimizer.zero_grad()` → forward trả logits **[B,10]**.
3. `CrossEntropyLoss` → `loss.backward()` → `optimizer.step()`.
4. Tổng hợp train loss, accuracy theo **số mẫu**; sau epoch đánh giá validation bằng `model.eval()`, không tính gradient.
5. Chọn/lưu checkpoint theo **validation accuracy cao nhất**, nếu bằng nhau thì validation loss thấp hơn.

**Cấu hình gốc:** `Adam`, `weight_decay=0.0001`, seed 42, chạy **CPU**, **1 epoch/run**. LR thử `0.001`, `0.0001`; batch thử **32**, **64**.

🎤 **Script:** “Training loop thực hiện forward, tính cross entropy, lan truyền ngược và cập nhật chỉ những tham số được mở. Sau mỗi epoch, nhóm validation trên dữ liệu không augmentation ngẫu nhiên và dùng kết quả này để chọn checkpoint. Test hoàn toàn không tham gia lựa chọn.”

**⚠** Train accuracy được tích lũy trong lúc weights đang thay đổi, val accuracy đo sau epoch; augmentation khác nhau. Không diễn giải chênh lệch hai số này như một phép so sánh tuyệt đối.


In [ ]:
show_function("src/train.py", "_run_epoch", limit=75)
print("Mã nguồn đầy đủ: src/train.py → train_model(); xem notebook 04 để chạy 5 thí nghiệm.")


## 5. Ma trận thực nghiệm — 1 phút

Không so sánh tùy tiện: mỗi cặp thay đổi **một yếu tố chính** trong điều kiện đã chốt.

- **E1 ↔ E2:** kiến trúc VGG16 vs DenseNet121, classifier-only.
- **E2 ↔ E3:** DenseNet121 classifier-only, **learning rate** 0.001 vs 0.0001.
- **E2 ↔ E4:** DenseNet121 classifier-only, **batch size** 32 vs 64.
- **E3 ↔ E5:** DenseNet121, LR 0.0001, batch 32, **classifier-only vs fine-tune-last-layers**.

🎤 **Script:** “Nhóm tổ chức 5 thí nghiệm để hiểu tác động của kiến trúc, learning rate, batch size và chiến lược fine-tuning. Những yếu tố khác được giữ giống nhau trong từng cặp đối chiếu khi có thể. Mỗi kết quả trong bảng được đọc từ file thực nghiệm đã lưu, không phải kết quả giả lập.”

**⚠** E5 cấu hình **10 epoch** trong CSV chưa có số liệu — không đưa vào bảng 5 run hoàn thành.


In [ ]:
cols = ["experiment_name", "model_name", "strategy",
        "learning_rate", "batch_size", "epochs", "seed", "val_accuracy_pct"]
display(results[cols].rename(columns={"val_accuracy_pct":"Validation accuracy (%)"}))
print("Lưu ý: dữ liệu đo sau 1 epoch/run trên CPU; E5_finetuning 10 epoch chưa có kết quả.")


## 6. Monitoring với TensorBoard — 2 phút

**Có 3 lớp theo dõi**:
- **Trong console:** tiến độ batch, loss/accuracy đang tích lũy.
- **Trong history/metrics:** `outputs/runs/<run>/history.csv`, `metrics.json` — lưu lại kết quả.
- **Trong TensorBoard:** bốn tag đã xác nhận ở run gốc: `Loss/train`, `Loss/val`, `Accuracy/train`, `Accuracy/val`.

Notebook `05_model_monitoring.ipynb` có thêm logic **batch logging, learning-rate log, generalization gap và cảnh báo sơ bộ** cho các lần chạy **full** mới. Những số liệu đó **không có sẵn cho các run lịch sử** nếu không được ghi trong log.

🎤 **Script:** “Monitoring giúp kiểm chứng thí nghiệm và phát hiện dấu hiệu bất thường. Trong phiên bản gốc, nhóm ghi train/val loss và accuracy cho 5 cấu hình. Vì mỗi run chỉ có một epoch, TensorBoard hiện chỉ là đối chiếu các điểm đo, chưa đủ dựng một learning curve có ý nghĩa hay kết luận overfitting.”

**Cách mở log trên máy nếu có event files thật:**
`python -m tensorboard.main --logdir outputs/runs --port 6006`
→ `http://localhost:6006`.
Với **chế độ review**, dùng ảnh TensorBoard lưu ở `reports/figures/tensorboard_comparison.png` làm minh chứng.


In [ ]:
show_figure("tensorboard_comparison.png", width=980)
rows=[]
for name in RUNS:
    path=OUT/"runs"/name/"history.csv"
    if path.is_file():
        h=pd.read_csv(path)
        if not h.empty:
            v=h.iloc[-1]
            rows.append({"Run":name,"Số epoch đã lưu":len(h),
                        "Train loss":v["train_loss"],"Val loss":v["val_loss"],
                        "Train acc (%)":100*v["train_accuracy"],
                        "Val acc (%)":100*v["val_accuracy"]})
history_table = pd.DataFrame(rows)
display(history_table.round(3))
if not history_table.empty:
    print("Các file history đọc từ ổ đĩa; không tạo ra dữ liệu epoch trung gian.")


### 6.1. Generalization gap: giải thích chính xác

**Accuracy gap (điểm phần trăm)** = `train_accuracy - val_accuracy`. **Loss gap** = `val_loss - train_loss`.

Nếu huấn luyện nhiều epoch và train loss liên tục giảm trong khi validation loss tăng, đó là một **dấu hiệu cần kiểm tra** overfitting, **không phải bằng chứng tuyệt đối**.

⚠ **Ở Lab 2 gốc:** 1 epoch/run → **không thể xác định hội tụ, xu hướng gap hay overfitting qua thời gian**. E5 có train accuracy thấp hơn validation là hiện tượng có thể xảy ra do train augmentation, dropout và thời điểm đo khác nhau.

🎤 **Script:** “Nhóm đã có cơ chế ghi chỉ số và có thể mở rộng monitoring cho nhiều epoch, nhưng em không suy diễn một learning curve từ chỉ một điểm trên biểu đồ. Đây cũng là một hạn chế khoa học cần công bố rõ.”


## 7. Kết quả E1–E5 & phân tích — 2 phút

**Luận điểm cần nói bằng số liệu thực đo:** so sánh validation accuracy và loss dưới **cùng phạm vi 1 epoch**, không dùng accuracy test để xếp hạng.

🎤 **Script:** “VGG16 classifier-only đạt 82,88%. DenseNet121 classifier-only ở LR 0,001 đạt 87,84%. Khi giảm LR còn 0,0001, accuracy còn 83,44%; khi tăng batch lên 64, accuracy đạt 87,16%. Cuối cùng, DenseNet121 fine-tuning các lớp cuối đạt cao nhất, 92,10%. Kết quả phản ánh các cấu hình đã thử, không chứng minh một hyperparameter luôn tốt hơn trong mọi hoàn cảnh.”

**Biểu đồ dưới đây đọc trực tiếp từ `experiment_summary.csv`, không vẽ dữ liệu minh họa.**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
labels=[x.split("_")[0] for x in results["experiment_name"]]
axes[0].bar(labels, results["val_accuracy_pct"])
axes[0].set(title="Validation accuracy (1 epoch)", ylabel="Accuracy (%)", ylim=(0,100))
axes[1].bar(labels, results["best_val_loss"])
axes[1].set(title="Validation loss (1 epoch)", ylabel="CrossEntropy")
for ax in axes:
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.2f", padding=3)
    ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


### 7.1. Các phép so sánh có kiểm soát và giới hạn

- **Kiến trúc:** E2 − E1 = **+4,96 điểm %** validation accuracy.
- **Learning rate:** E2 − E3 = **+4,40 điểm %** ở classifier-only sau 1 epoch.
- **Batch size:** E2 − E4 = **+0,68 điểm %**; E4 không nhanh hơn E2 trong hai lượt đo này.
- **Chiến lược:** E5 − E3 = **+8,66 điểm %** ở cùng LR 0.0001, batch 32.

🎤 **Script:** “Bốn cặp so sánh cung cấp bằng chứng định lượng để trả lời đúng yêu cầu điều chỉnh kiến trúc và hyperparameter. Tuy nhiên, batch khác nhau kéo theo số optimizer steps khác nhau trong cùng một epoch. Do đó nhóm chỉ kết luận trong phạm vi đã thử.”

**Câu hỏi thầy:** “Tại sao LR nhỏ hơn lại kém hơn?” → “Trong một epoch và cấu hình classifier-only cụ thể, LR nhỏ có thể chưa cập nhật đủ nhanh; để kết luận tổng quát cần nhiều epoch/seed hơn.”


In [ ]:
r=results.set_index("experiment_name")
comparisons=[
    ("Model (E2 - E1)", "E2_densenet121", "E1_vgg16"),
    ("LR (E2 - E3)", "E2_densenet121", "E3_learning_rate"),
    ("Batch (E2 - E4)", "E2_densenet121", "E4_batch_size"),
    ("Fine-tuning (E5 - E3)", "E5_cpu_1epoch", "E3_learning_rate"),
]
diff=pd.DataFrame([{"So sánh":label,
                   "Chênh lệch validation (điểm %)":
                       r.loc[a,"val_accuracy_pct"]-r.loc[b,"val_accuracy_pct"]}
                   for label,a,b in comparisons])
display(diff.round(2))
display(results[["experiment_name","duration_minutes"]].rename(
    columns={"duration_minutes":"Training + validation (phút)"}).round(1))


## 8. Chọn mô hình và đánh giá test — 2 phút

**Nguyên tắc:** xếp hạng bằng `best_val_accuracy` **giảm dần**, nếu bằng nhau thì `best_val_loss` **tăng dần**. Chọn xong mới đọc **test**.

Theo kết quả gốc: **E5_cpu_1epoch** là DenseNet121 `finetune_last_layers`, LR `0.0001`, batch `32`, seed `42`. Validation accuracy **92,10%**, validation loss **0,232135**.

**Test checkpoint E5 trên 10.000 ảnh:** accuracy **91,81%**, test loss **0,245938**, weighted precision **0,918185**, weighted recall **0,918100**, weighted F1 **0,917776**.

🎤 **Script:** “Chúng em không dùng test để quyết định hyperparameter. E5 được chọn hoàn toàn trên validation, sau đó mới nạp checkpoint và đánh giá 10.000 ảnh test bằng `model.eval()` và `torch.no_grad()`. Kết quả test 91,81% gần với validation 92,10%, chênh 0,29 điểm phần trăm.”

**⚠ Nguồn test:** số đo được ghi lại từ **terminal output đã làm tròn 6 chữ số** trong `metrics.json`; không tuyên bố notebook này vừa tính lại test hoặc đã xác thực checkpoint đang tồn tại trên GitHub.


In [ ]:
best = results.sort_values(["best_val_accuracy","best_val_loss"],
                           ascending=[False,True]).iloc[0]
assert best["experiment_name"] == "E5_cpu_1epoch", "Best run đã thay đổi, xem lại script!"
m_path=OUT/"runs"/best["experiment_name"]/"metrics.json"
assert m_path.is_file(), f"Thiếu {m_path}"
metrics=json.loads(m_path.read_text(encoding="utf-8"))
final_test=metrics.get("final_test", {})
print("ĐƯỢC CHỌN THEO VALIDATION:", best["experiment_name"])
print(f"Validation accuracy: {best['best_val_accuracy']:.2%}")
print(f"Validation loss: {best['best_val_loss']:.6f}")
print("Nguồn test:", final_test.get("source","không rõ"))
report_metrics=pd.DataFrame([{
    "Test samples":final_test.get("test_samples"),
    "Test loss":final_test.get("test_loss"),
    "Accuracy (%)":100*final_test.get("accuracy", float("nan")),
    "Precision weighted":final_test.get("precision"),
    "Recall weighted":final_test.get("recall"),
    "F1 weighted":final_test.get("f1_score"),
}])
display(report_metrics.round(4))
print("Validation - test (điểm %):",
      round(100*(best["best_val_accuracy"]-final_test.get("accuracy",float("nan"))),2))


## 9. Demo có thể tái lập & kiểm tra code — 1 phút

**Demo không tốn nhiều giờ:**
1. Trình bày `src/data.py`: split, train/eval transforms.
2. Trình bày `src/models.py`: `_replace_classifier`, `_configure_trainable_parameters` và shape `[B,10]`.
3. Trình bày `src/train.py`: forward / loss / backward / step, validation.
4. Mở bằng chứng 5 run, TensorBoard screenshot, chọn E5 và đọc test JSON.
5. Chỉ **nếu có đủ môi trường và thời gian**, chạy cell kiểm tra shape bên dưới; nó **không nạp weights ImageNet**, chỉ chứng minh kiến trúc có output 10 logits.

**Lưu ý:** `main.py` sử dụng YAML để chạy pipeline thực tế; các notebook 01–06 có chế độ `full` chạy mới. Notebook trình bày này **không tự chạy thí nghiệm**, tránh ghi đè output gốc.

🎤 **Script:** “Để đảm bảo kết quả có thể đối chiếu, mỗi run có config, history, metrics và đường dẫn checkpoint tương ứng. Notebook này đọc trực tiếp những artifact đó; các notebook chi tiết và `src/` giữ phần hiện thực đầy đủ.”


In [ ]:
RUN_OPTIONAL_SHAPE_CHECK = False  # Đổi True chỉ khi đã cài torch/torchvision và muốn demo trên máy
if RUN_OPTIONAL_SHAPE_CHECK:
    import torch
    from src.models import build_model
    cfg={"model_name":"densenet121","pretrained":False,
         "strategy":"classifier_only","num_classes":10}
    model=build_model(cfg).eval()
    with torch.no_grad():
        logits=model(torch.zeros(1,3,224,224))
    assert logits.shape==(1,10)
    print("Shape kiểm tra thành công:", tuple(logits.shape))
    print("Đây là shape-check với pretrained=False, KHÔNG phải evaluation accuracy.")
else:
    print("Bỏ qua kiểm tra shape live; không tải weights và không train.")


## 10. Kết luận, hạn chế và hướng phát triển — 1 phút

**Kết quả Lab 2 đạt được:** đã thử **2 kiến trúc pre-trained**, thay classifier 10 lớp, sử dụng classifier-only/fine-tuning, xử lý CIFAR-10, hoàn thành **5 lần chạy**, ghi train/val metrics, theo dõi TensorBoard và chọn checkpoint theo validation. **Test accuracy E5 = 91,81%.**

**Giới hạn bắt buộc nói rõ:**
- **1 epoch / 1 seed / run**, CPU → chưa kết luận convergence hay tính ổn định.
- **Chưa có confusion matrix** và phân tích lỗi theo lớp chính thức.
- Chưa có kết quả cho **E5 10 epoch**.
- Kết quả monitoring gốc chỉ ở cấp **epoch**, khác log bổ sung của notebook mới khi chạy `full`.
- Checkpoint, dataset, TensorBoard event files lớn có thể **không nằm trên GitHub**; cần giữ bản lưu cục bộ để tái lập.

**Đề xuất phát triển:** chạy nhiều epoch/seed trên GPU khả dụng; thêm confusion matrix, per-class accuracy; so sánh learning curves và thời gian chuẩn hóa.

🎤 **Script kết:** “Lab 2 giúp nhóm hiểu toàn bộ quá trình chuyển giao học từ mô hình pre-trained sang bộ dữ liệu mới, lựa chọn chiến lược huấn luyện dựa trên validation và đánh giá test đúng quy trình. Kết quả 91,81% test là kết quả thực nghiệm của nhóm trong phạm vi đã chạy, không phải tuyên bố mô hình đã tối ưu hay hội tụ.”


## 11. Bộ câu hỏi phản biện nhanh

**Q1 — Tại sao không training VGG16/DenseNet121 từ đầu?**  
A — Đề yêu cầu pre-trained architectures; ImageNet weights cung cấp biểu diễn thị giác sẵn có để chuyển giao.

**Q2 — Classifier-only khác fine-tuning thế nào?**  
A — Classifier-only freeze backbone và train classifier; fine-tuning E5 mở thêm denseblock4/norm5.

**Q3 — Ảnh CIFAR-10 32×32, tại sao resize 224×224?**  
A — Phù hợp preprocessing sử dụng với mô hình ImageNet pre-trained; resize không tự tạo thêm thông tin gốc.

**Q4 — Softmax đặt ở đâu?**  
A — `CrossEntropyLoss` nhận logits thô, nên không thêm Softmax trước loss.

**Q5 — Làm sao chứng minh monitoring hoạt động?**  
A — `history.csv`, `metrics.json` và screenshot TensorBoard bốn tag; muốn xem TensorBoard tương tác cần event logs cục bộ.

**Q6 — 1 epoch có biết overfit không?**  
A — Chưa. Cần chuỗi train/validation qua nhiều epoch và/hoặc nhiều seed.

**Q7 — Vì sao E5 được chọn?**  
A — Validation accuracy cao nhất 92,10%; test chỉ dùng sau khi chốt checkpoint.

**Q8 — Có thật chạy 10 epoch không?**  
A — Không; E5 cấu hình 10 epoch chưa có metrics nên nhóm không tính là thí nghiệm hoàn thành.

**Q9 — PCA đâu?**  
A — PCA/EDA nâng cao là nghiên cứu bổ sung ở công việc trước; nhánh `feature/data` hiện chỉ giữ EDA trong notebook 03; PCA không phải core đầu ra Lab 2.

**Q10 — Notebook tổng hợp có thay code module gốc không?**  
A — Không. Nó trình bày và kiểm chứng artifact đã lưu. Code huấn luyện thật vẫn nằm trong `src/` và notebook 04.


## 12. Chỉ mục để giảng viên kiểm tra mã nguồn

| Câu hỏi | Nơi đối chiếu |
|---|---|
| Dataset, transform, split | `src/data.py`, `notebooks/03_data_exploration_preprocessing.ipynb` |
| Pretrained architecture | `notebooks/01_environment_pretrained.ipynb`, `src/models.py` |
| Classifier, freeze, fine-tune | `src/models.py`, `notebooks/02_model_adaptation.ipynb` |
| Training và checkpoint | `src/train.py`, `notebooks/04_training_experiments.ipynb` |
| Monitoring/TensorBoard | `notebooks/05_model_monitoring.ipynb`, `reports/figures/tensorboard_comparison.png` |
| 5 run và hyperparameters | `outputs/experiment_summary.csv`, `configs/*.yaml` |
| Test E5 | `outputs/runs/E5_cpu_1epoch/metrics.json`, `src/evaluate.py` |
| Báo cáo khoa học | `reports/Lab2_Report.md` |

**Kết thúc:** Mở phần bảng thực nghiệm hoặc hình TensorBoard để trả lời câu hỏi; không chạy lại training tại buổi present khi không cần thiết.

**Thông tin provenance:** Được tổng hợp từ các nguồn repository đã audit ngày 20/11/2026, nhánh `feature/data`; các con số mô tả kết quả lịch sử, không phải kết quả vừa tái chạy.
